# Kaggle: predict + submit (0.942-pipeline port)

Attach as Datasets: the competition data, `cell-tracking-src` (the zip from `scripts/package_for_kaggle.py`: code + `models/primary.pth`, `models/secondary.pth`, `models/deepcenter.pt` + `ARTIFACT_MANIFEST.json`), and `biohub-tracking-wheels` (= `context/pack_primary/wheels/`, the offline wheels for tracksdata / ilpy / pyscipopt / polars / ...). This notebook runs with internet OFF. Needs a GPU accelerator.

In [ ]:
import subprocess, sys
from pathlib import Path

# Mount names of the attached Datasets (2026-09-16: kaigow/cell-tracking-0942-{wheels,src}); adjust if renamed.
WHEELS_DATASET = Path('/kaggle/input/datasets/kaigow/cell-tracking-0942-wheels')
SRC_DATASET = Path('/kaggle/input/datasets/kaigow/cell-tracking-0942-src')
assert SRC_DATASET.exists(), f'code Dataset not attached at {SRC_DATASET}'
assert WHEELS_DATASET.exists(), f'wheels Dataset not attached at {WHEELS_DATASET}'

wheel_dirs = sorted({p.parent for p in WHEELS_DATASET.rglob('*.whl')})
assert wheel_dirs, f'no .whl files under {WHEELS_DATASET}'
# Browser uploads/downloads have truncated wheels above ~9 MB before (2026-09-16: pip 'Wheel ... is invalid').
# Fail here with the offending files instead of deep inside pip.
import zipfile
bad = []
for w in sorted(WHEELS_DATASET.rglob('*.whl')):
    try:
        with zipfile.ZipFile(w) as zf:
            if zf.testzip() is not None:
                bad.append(w.name)
    except zipfile.BadZipFile:
        bad.append(w.name)
assert not bad, f'{len(bad)} truncated/corrupt wheel(s) in the Dataset -- re-upload it: {bad}'
print('all', sum(1 for _ in WHEELS_DATASET.rglob('*.whl')), 'wheels are valid zips')
# Kaggle strips '+' from uploaded filenames (tracksdata-0.1.0rc6.dev3+g980c2d30a -> ...dev3g980c2d30a), which
# pip then ignores as an invalid version. Copy every wheel to a writable dir under the name its own
# METADATA dictates ({name}-{version}-{tags}.whl).
import shutil
FIXED = Path('/kaggle/working/wheels'); FIXED.mkdir(parents=True, exist_ok=True)
def canonical_wheel_name(w: Path) -> str:
    with zipfile.ZipFile(w) as zf:
        meta = next(n for n in zf.namelist() if n.endswith('.dist-info/METADATA'))
        fields = dict(l.split(': ', 1) for l in zf.read(meta).decode().split('\n\n', 1)[0].splitlines() if ': ' in l)
    name, _, tags = w.stem.split('-', 2)   # keep the file's name segment; only the version can be mangled
    return f"{name}-{fields['Version']}-{tags}.whl"
renamed = []
for w in sorted(WHEELS_DATASET.rglob('*.whl')):
    target = FIXED / canonical_wheel_name(w)
    if target.name != w.name:
        renamed.append(f'{w.name} -> {target.name}')
    shutil.copyfile(w, target)
print('wheels staged in', FIXED, '| renamed:', renamed or 'none')
find_links = ['--find-links', str(FIXED)]
# Offline install of the ILP stack (+ blosc2/zarr if the image lacks them). Do not
# install the bundled geff wheel if a newer geff is preinstalled.
pkgs = ['tracksdata', 'ilpy', 'pyscipopt', 'polars', 'polars_runtime_32', 'rustworkx',
        'sqlalchemy', 'dask', 'imagecodecs', 'pyarrow', 'blosc2', 'zarr']
# Keep the image's numpy/scipy/torch: pip otherwise upgrades numpy to the newest wheel on offer, and the
# kernel already holds the old numpy in memory -> 'cannot import name _center from numpy._core.umath'
# (2026-09-16). Nothing in the wheel set needs a newer numpy (tracksdata: numpy>2, numba 0.65: numpy<2.5).
import importlib.metadata as md
pins = []
for pkg in ('numpy', 'scipy', 'torch'):
    try:
        pins.append(f'{pkg}=={md.version(pkg)}')
    except md.PackageNotFoundError:
        pass
CONSTRAINTS = Path('/kaggle/working/constraints.txt'); CONSTRAINTS.write_text('\n'.join(pins) + '\n')
print('pip constraints:', pins)
base = [sys.executable, '-m', 'pip', 'install', '--no-index', *find_links]
r = subprocess.run([*base, '-c', str(CONSTRAINTS), *pkgs])
if r.returncode != 0:
    print('constrained install failed (rc', r.returncode, ') -- retrying without the pins')
    subprocess.run([*base, *pkgs], check=True)
import os
os.environ['POLARS_PREFER_PKG'] = '32'
# Import check in a fresh interpreter: this kernel must not import numpy-dependent packages after pip
# touched site-packages. predict.py / make_submission.py below run as subprocesses for the same reason.
subprocess.run([sys.executable, '-c',
                'import tracksdata, ilpy, pyscipopt, blosc2, zarr, numpy, scipy, torch; '
                'print("ILP stack ok:", tracksdata.__version__, "| numpy", numpy.__version__, '
                '"scipy", scipy.__version__, "torch", torch.__version__)'],
               check=True, env={**os.environ, 'POLARS_PREFER_PKG': '32'})


In [ ]:
import json, os
sys.path.insert(0, str(SRC_DATASET / 'src'))
SRC_ENV = {**os.environ, 'PYTHONPATH': str(SRC_DATASET / 'src'), 'POLARS_PREFER_PKG': '32'}

manifest = json.loads((SRC_DATASET / 'ARTIFACT_MANIFEST.json').read_text())
print('artifact manifest:', json.dumps(manifest, indent=1)[:3000])
models = manifest['models']
assert 'primary' in models, 'no primary model in this artifact'
PRIMARY = SRC_DATASET / models['primary']['path']
SECONDARY = SRC_DATASET / models['secondary']['path'] if 'secondary' in models else None
DEEPCENTER = SRC_DATASET / models['deepcenter']['path'] if 'deepcenter' in models else None
for p in (PRIMARY, SECONDARY, DEEPCENTER):
    assert p is None or p.exists(), p
import hashlib
for key, entry in models.items():
    digest = hashlib.sha256((SRC_DATASET / entry['path']).read_bytes()).hexdigest()
    assert digest == entry['sha256'], f'{key}: sha256 mismatch -- wrong weights attached'
print('weights verified:', {k: v['sha256'][:12] for k, v in models.items()})


In [ ]:
# In a subprocess for the same reason as above (the kernel must not import numpy after the pip install).
subprocess.run([sys.executable, '-c',
                'from cell_tracking import config; '
                'print("on_kaggle:", config.on_kaggle()); print("test_dir:", config.get_test_dir())'],
               check=True, env=SRC_ENV)


In [ ]:
cmd = [sys.executable, str(SRC_DATASET / 'scripts' / 'predict.py'),
       '--checkpoint', str(PRIMARY), '--out-dir', '/kaggle/working/preds',
       '--dump-stats', '/kaggle/working/predict_stats.json']
if SECONDARY is not None:
    cmd += ['--secondary-checkpoint', str(SECONDARY)]
if DEEPCENTER is not None:
    cmd += ['--deepcenter', str(DEEPCENTER)]
    # predict.py asserts the pack's DeepCenter epoch (2) by default; our own trainer's best.pt
    # is a different epoch. The manifest sha256 check above already pins the file.
    if models['deepcenter'].get('epoch') != 2:
        cmd += ['--no-deepcenter-epoch-check']
print(' '.join(cmd))
subprocess.run(cmd, check=True, env=SRC_ENV)


In [ ]:
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'make_submission.py'),
                '--geff-dir', '/kaggle/working/preds', '--out', 'submission.csv'], check=True, env=SRC_ENV)


In [ ]:
# Schema / topology audit (the 0.942 notebook's rules): contiguous ids, dt=1, in-degree<=1, out-degree<=2, no negatives.
import csv
from collections import defaultdict
rows = list(csv.DictReader(open('submission.csv')))
assert [int(r['id']) for r in rows] == list(range(len(rows))), 'ids not contiguous'
by_ds = defaultdict(lambda: {'t': {}, 'in': defaultdict(int), 'out': defaultdict(int)})
for r in rows:
    d = by_ds[r['dataset']]
    if r['row_type'] == 'node':
        assert min(int(r['t']), int(r['z']), int(r['y']), int(r['x'])) >= 0, r
        d['t'][int(r['node_id'])] = int(r['t'])
for r in rows:
    if r['row_type'] == 'edge':
        d = by_ds[r['dataset']]
        s, t = int(r['source_id']), int(r['target_id'])
        assert s in d['t'] and t in d['t'], 'dangling edge'
        assert d['t'][t] == d['t'][s] + 1, 'edge not dt=1'
        d['in'][t] += 1; d['out'][s] += 1
for name, d in by_ds.items():
    assert max(d['in'].values(), default=0) <= 1, f'{name}: in-degree > 1'
    assert max(d['out'].values(), default=0) <= 2, f'{name}: out-degree > 2'
print('audit ok:', len(rows), 'rows,', len(by_ds), 'datasets')
